# Flood Risk Prediction using XGBoost
**Author:** Ale Navarro

This notebook trains an XGBoost classifier using the shared `common.py` module so it is directly comparable with the other models in the project.


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import joblib
import matplotlib.pyplot as plt
import common

from xgboost import XGBClassifier
from sklearn.metrics import ConfusionMatrixDisplay, RocCurveDisplay

## 1. Load data

In [ ]:
df = common.load_data()

X, y = common.build_features(df)

X_train, X_test, y_train, y_test = common.chronological_split(X, y)

print(f"Training samples: {len(X_train)}")
print(f"Testing samples: {len(X_test)}")

## 2. Train XGBoost

In [ ]:
model = XGBClassifier(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42
)

model.fit(X_train, y_train)

## 3. Predictions

In [ ]:
preds = model.predict(X_test)
probs = model.predict_proba(X_test)[:,1]

## 4. Evaluate

In [ ]:
baseline = common.persistence_baseline(df, y_test)

results = [
    baseline,
    common.evaluate("XGBoost", y_test, preds, probs)
]

common.comparison_table(results)

## 5. Confusion Matrix

In [ ]:
ConfusionMatrixDisplay.from_predictions(y_test, preds)
plt.show()

## 6. ROC Curve

In [ ]:
RocCurveDisplay.from_predictions(y_test, probs)
plt.show()

## 7. Feature Importance

In [ ]:
importance = sorted(
    zip(X.columns, model.feature_importances_),
    key=lambda x: x[1],
    reverse=True
)

for feature, score in importance:
    print(f"{feature}: {score:.4f}")

## 8. Save model

In [ ]:
joblib.dump(model, "../models/xgboost.joblib")
print("Model saved to ../models/xgboost.joblib")

## Conclusions

- Uses the shared preprocessing from `common.py`.
- Uses the agreed chronological split.
- Evaluated with the common metrics for fair comparison.
- Saved as `xgboost.joblib` for dashboard integration.
